In [ ]:
import os
import sys

original_sys_path = sys.path.copy()

try:
    import module
except ImportError:
    print('Module system not found. Make sure the required python packages are available in the current environment.')
else:
    sys.path = original_sys_path
    await module.purge(force=True)
    print('Loading required modules from EESSI...')
    await module.load('AITW-microstructures/1.0.0-foss-2023a')

    import os
    import sys

    for extra in os.getenv('PYTHONPATH').split(':'):
        if extra not in sys.path:
            sys.path.append(extra)

In [ ]:
import rich
import ipywidgets as wdg
from wood_microstructure.cli.generate import generate

from common import WOOD_MS_DEFAULTS

import utilities

console = rich.get_console()
console.is_jupyter = True
console.width = 140

common_event: wdg.IntText = wdg.IntText(
    value=0, description='', disabled=True,
)

widgets_launch = utilities.get_widgets_from_click_function(
    generate,
    override_defaults=WOOD_MS_DEFAULTS,
)

title = wdg.HTML(value="<h2>Microstructure Generation Parameters</h2>")
display(title)
widget_box = wdg.VBox(list(widgets_launch.values()))
display(widget_box)

generate_button = wdg.Button(description="Generate Microstructure", button_style="success")
output_area = wdg.Output()

@utilities.widget_button_running(running_text="Generating...")
def on_generate_button_clicked(b):
    with output_area:
        output_area.clear_output()
        args = utilities.get_click_args_from_widgets(generate, widgets_launch)
        generate(args, standalone_mode=False)
    common_event.value += 1

generate_button.on_click(on_generate_button_clicked)
display(generate_button)
display(output_area)

In [ ]:
# Rewrite of the above cell with autodetection of which folders contains the images
import re
import os
import json
from IPython.display import display
from IPython.display import Image as IPyImage

rgx_dir = re.compile(r"Save(\w+)_(\d+)")
rgx_img = re.compile(r"volImgRef_(\d+).png")

title = wdg.HTML(value="<h2>Microstructure Output Visualization</h2>")

def find_outputs(base_path="../results/wood_ms_results/"):
    outputs = []
    if not os.path.exists(base_path):
        return outputs
    for path in os.listdir(base_path):
        if not os.path.isdir(os.path.join(base_path, path)):
            continue
        match = rgx_dir.match(path)
        if not match:
            continue
        material, index = match.groups()
        outputs.append((material, int(index), os.path.join(base_path, path)))

    return outputs

# Start with a choice widget to select which output to visualize
output_selector = wdg.Dropdown(
    options=[],
    description='Select Output:',
    disabled=False,
    style={'description_width': '150px'},
)
output_selector.layout.width = '500px'

def update_output_options(*args):
    """Update the options of the output selector based on the available outputs in the results folder."""
    outputs = find_outputs()
    options = []
    for material, index, path in outputs:
        params = json.load(open(os.path.join(path, "params.json")))
        surrogate = params['surrogate']
        with_ray_cell = bool(params['is_exist_ray_cell'])
        with_vessel = bool(params['is_exist_vessel'])
        options.append((
            f"{material} - {index}: surrogate={surrogate}, ray_cells={with_ray_cell}, vessels={with_vessel}",
            path
        ))
    options.sort(key=lambda x: x[0])
    output_selector.options = options
    if output_selector.value is None:
        output_selector.value = options[0][1] if options else None

# Observe changes in the common event to update the output options whenever a new microstructure is generated
try:
    common_event.observe(update_output_options, names='value')
except Exception as e:
    print("common-event not defined.")
    print("Re-launch this cell after executing the previous one to enable automatic updates of the dropdown choices.")

# Select whether to show the local distortion images or the backbone images, and a slider to select which slice to show
toggle_checkbox = wdg.Checkbox(
    value=False,
    description='With local distortion',
    style={'description_width': '150px'},
)
toggle_checkbox.layout.width = '500px'

# Slider to select which slice to show, with options automatically updated based on the selected output
image_slider = wdg.SelectionSlider(
    options=[0],
    description='Slice number:',
    continuous_update=False,
    style={'description_width': '150px'},
)
image_slider.layout.width = '500px'

image_output = wdg.Output()

def update_image_options(*args):
    """Update the options of the image slider based on the selected output and toggle state."""
    selected_output = output_selector.value
    if not selected_output:
        return
    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    custom_index = []
    for file in os.listdir(os.path.join(selected_output, imgdir)):
        match = rgx_img.match(file)
        if match:
            custom_index.append(int(match.group(1)))
    custom_index.sort()

    image_slider.options = custom_index
    image_slider.value = image_slider.options[0] if image_slider.options else None
output_selector.observe(update_image_options, names='value')

def display_image(*args):
    """Display the selected image based on the selected output, toggle state, and slider value."""
    selected_output = output_selector.value
    if not selected_output:
        return

    imgdir = "LocalDistVolume" if toggle_checkbox.value else "volImgBackBone"
    index_list = image_slider.options
    image_list = [os.path.join(selected_output, imgdir, f"volImgRef_{idx:05d}.png") for idx in index_list]

    selected_custom_index = image_slider.value
    position = index_list.index(selected_custom_index)

    with image_output:
        image_output.clear_output(wait=True)
        display(IPyImage(url=image_list[position]))

# Observe changes in the toggle and slider to update the displayed image
toggle_checkbox.observe(display_image, names='value')
image_slider.observe(display_image, names='value')
output_selector.observe(display_image, names='value')

# Initial update of the output options and image options
update_output_options()

display(title, output_selector, toggle_checkbox, image_slider, image_output)